In [1]:
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path

# Load raw data
races = pd.read_csv("../data/raw/races.csv")
results = pd.read_csv("../data/raw/results.csv")
lap_times = pd.read_csv("../data/raw/lap_times.csv")
circuits = pd.read_csv("../data/raw/circuits.csv")

print("✓ Data loaded")

✓ Data loaded


In [2]:
# Load circuit baselines created in Notebook 02
print("\nLoading circuit baselines (from Notebook 02)...")

with open("../data/processed/circuit_baselines.json", "r") as f:
    circuit_baselines_by_name = json.load(f)

# Create lookup: circuitId -> stats
circuit_baselines = {}
for circuit_id in circuits['circuitId'].unique():
    circuit_name = circuits[circuits['circuitId'] == circuit_id]['name'].values[0]
    if circuit_name in circuit_baselines_by_name:
        circuit_baselines[circuit_id] = circuit_baselines_by_name[circuit_name]

print(f"✓ Loaded baselines for {len(circuit_baselines)} circuits")
print(f"\nSample baselines:")
for circuit_name in list(circuit_baselines_by_name.keys())[:3]:
    stats = circuit_baselines_by_name[circuit_name]
    print(f"  {circuit_name:40s}: mean={stats['mean']/1000:6.2f}s, std={stats['std']/1000:5.2f}s")


Loading circuit baselines (from Notebook 02)...
✓ Loaded baselines for 41 circuits

Sample baselines:
  Albert Park Grand Prix Circuit          : mean= 98.99s, std=65.05s
  Sepang International Circuit            : mean=110.05s, std=108.44s
  Bahrain International Circuit           : mean= 99.78s, std=39.03s


In [3]:
# STEP 1: Recreate features exactly as in Notebook 02

print("\n" + "="*70)
print("STEP 1: RECREATE FEATURES FROM NOTEBOOK 02")
print("="*70)

lap_times_clean = lap_times[lap_times['milliseconds'].notna()].copy()

features_list = []
for (race_id, driver_id), group in lap_times_clean.groupby(['raceId', 'driverId']):
    early_laps = group[group['lap'] <= 10].copy()
    if len(early_laps) < 5:
        continue
    
    laps_completed_early = len(early_laps)
    avg_lap_time = early_laps['milliseconds'].mean()
    lap_time_std = early_laps['milliseconds'].std()
    lap_time_min = early_laps['milliseconds'].min()
    
    if len(early_laps) >= 6:
        first_3 = early_laps.head(3)['milliseconds'].mean()
        last_3 = early_laps.tail(3)['milliseconds'].mean()
        pace_degradation = last_3 - first_3
    else:
        pace_degradation = 0
    
    avg_position_early = early_laps['position'].mean()
    
    features_list.append({
        'raceId': race_id,
        'driverId': driver_id,
        'laps_completed_early': laps_completed_early,
        'avg_lap_time': avg_lap_time,
        'lap_time_std': lap_time_std,
        'lap_time_min': lap_time_min,
        'pace_degradation': pace_degradation,
        'avg_position_early': avg_position_early
    })

features_df = pd.DataFrame(features_list)
print(f"\n✓ Features extracted: {features_df.shape}")


STEP 1: RECREATE FEATURES FROM NOTEBOOK 02

✓ Features extracted: (10556, 8)


In [4]:
# STEP 2: Add circuit features

print("\n" + "="*70)
print("STEP 2: ADD CIRCUIT-AWARE FEATURES (Z-SCORE)")
print("="*70)

# Merge with races to get circuitId
features_with_circuit = features_df.merge(races[['raceId', 'circuitId']], on='raceId', how='left')

# Get circuit names for reference
features_with_circuit = features_with_circuit.merge(
    circuits[['circuitId', 'name']], 
    on='circuitId', 
    how='left'
)

print(f"\nMerged with circuits: {features_with_circuit.shape}")

print("\nCreating NEW feature: avg_lap_time_relative_zscore")
print("Formula: (driver_lap_time - circuit_mean) / circuit_std")
print("\nThis tells the model how many standard deviations away from circuit average")

# NEW FEATURE: Z-score relative to circuit average
features_with_circuit['avg_lap_time_relative_zscore'] = features_with_circuit.apply(
    lambda row: (
        (row['avg_lap_time'] - circuit_baselines[row['circuitId']]['mean']) / 
        circuit_baselines[row['circuitId']]['std']
    ) if row['circuitId'] in circuit_baselines else 0,
    axis=1
)

# OLD FEATURE: Normalized circuit ID (keep for context)
circuit_mean = features_with_circuit['circuitId'].mean()
circuit_std = features_with_circuit['circuitId'].std()
features_with_circuit['circuit_normalized'] = (features_with_circuit['circuitId'] - circuit_mean) / circuit_std

print("\n✓ Circuit features created")
print("\nExample: Z-score relative lap times")
sample = features_with_circuit[[
    'name', 
    'avg_lap_time', 
    'avg_lap_time_relative_zscore'
]].head(10).copy()
sample['avg_lap_time_sec'] = sample['avg_lap_time'] / 1000
sample = sample[['name', 'avg_lap_time_sec', 'avg_lap_time_relative_zscore']]
print(sample.to_string(index=False))


STEP 2: ADD CIRCUIT-AWARE FEATURES (Z-SCORE)

Merged with circuits: (10556, 10)

Creating NEW feature: avg_lap_time_relative_zscore
Formula: (driver_lap_time - circuit_mean) / circuit_std

This tells the model how many standard deviations away from circuit average

✓ Circuit features created

Example: Z-score relative lap times
                          name  avg_lap_time_sec  avg_lap_time_relative_zscore
Albert Park Grand Prix Circuit           93.1190                     -0.090190
Albert Park Grand Prix Circuit           99.3242                      0.005201
Albert Park Grand Prix Circuit           91.8885                     -0.109106
Albert Park Grand Prix Circuit           93.8651                     -0.078720
Albert Park Grand Prix Circuit           92.4852                     -0.099933
Albert Park Grand Prix Circuit           97.0215                     -0.030198
Albert Park Grand Prix Circuit           94.4683                     -0.069448
Albert Park Grand Prix Circuit       

In [5]:
# Merge with race results to get actual finish positions
print("\nMerging with race results...")

race_results = results[['raceId', 'driverId', 'positionText', 'points']].copy()
race_results['position_numeric'] = pd.to_numeric(race_results['positionText'], errors='coerce')
race_results = race_results[race_results['position_numeric'].notna()].copy()

df = features_with_circuit.merge(race_results, on=['raceId', 'driverId'], how='inner')

# Create target variables
df['top_10'] = (df['position_numeric'] <= 10).astype(int)

print(f"✓ Merged dataset: {df.shape}")
print(f"\nTarget distribution:")
print(f"  Top 10 finishes: {df['top_10'].sum()} ({100*df['top_10'].mean():.1f}%)")


Merging with race results...
✓ Merged dataset: (8468, 16)

Target distribution:
  Top 10 finishes: 5270 (62.2%)


In [6]:
# STEP 3: Extract features for model

print("\n" + "="*70)
print("STEP 3: EXTRACT FEATURES FOR NEURAL NETWORK")
print("="*70)

# NEW: Use avg_lap_time_relative_zscore instead of raw avg_lap_time
feature_columns = [
    'laps_completed_early',
    'avg_lap_time_relative_zscore',  # NEW: Z-score relative to circuit (replacing raw avg_lap_time)
    'lap_time_std',
    'lap_time_min',
    'pace_degradation',
    'avg_position_early',
    'circuit_normalized'
]

X = df[feature_columns].values.astype(np.float32)
y = df['top_10'].values.astype(np.float32).reshape(-1, 1)

print(f"\nFeature columns ({len(feature_columns)}):")
for i, col in enumerate(feature_columns, 1):
    marker = " ← NEW (Z-score relative to circuit avg)" if col == 'avg_lap_time_relative_zscore' else ""
    print(f"  {i}. {col}{marker}")

print(f"\nRaw features shape: {X.shape}")
print(f"Target shape: {y.shape}")

print(f"\nFeature statistics (before normalization):")
for i, col in enumerate(feature_columns):
    print(f"  {col:35s}: mean={X[:, i].mean():8.2f}, std={X[:, i].std():8.2f}")


STEP 3: EXTRACT FEATURES FOR NEURAL NETWORK

Feature columns (7):
  1. laps_completed_early
  2. avg_lap_time_relative_zscore ← NEW (Z-score relative to circuit avg)
  3. lap_time_std
  4. lap_time_min
  5. pace_degradation
  6. avg_position_early
  7. circuit_normalized

Raw features shape: (8468, 7)
Target shape: (8468, 1)

Feature statistics (before normalization):
  laps_completed_early               : mean=   10.00, std=    0.06
  avg_lap_time_relative_zscore       : mean=    0.21, std=    0.71
  lap_time_std                       : mean=26042.12, std=110311.86
  lap_time_min                       : mean=92922.42, std=12887.42
  pace_degradation                   : mean=-17572.67, std=116370.78
  avg_position_early                 : mean=   10.13, std=    5.75
  circuit_normalized                 : mean=    0.03, std=    1.02


In [7]:
# STEP 4: Split BEFORE normalizing to prevent data leakage

print("\n" + "="*70)
print("STEP 4: SPLIT DATA (BEFORE NORMALIZATION)")
print("="*70)
print("\nSplitting: 60% train, 20% val, 20% test")

np.random.seed(42)
n_samples = len(X)
indices = np.random.permutation(n_samples)

# 60% train, 20% val, 20% test
train_split = int(0.6 * n_samples)
val_split = int(0.8 * n_samples)

train_idx = indices[:train_split]
val_idx = indices[train_split:val_split]
test_idx = indices[val_split:]

X_train_raw = X[train_idx]
y_train = y[train_idx]

X_val_raw = X[val_idx]
y_val = y[val_idx]

X_test_raw = X[test_idx]
y_test = y[test_idx]

print(f"\nTraining set:   {X_train_raw.shape} - {np.sum(y_train):.0f} top-10 ({100*np.mean(y_train):.1f}%)")
print(f"Validation set: {X_val_raw.shape} - {np.sum(y_val):.0f} top-10 ({100*np.mean(y_val):.1f}%)")
print(f"Test set:       {X_test_raw.shape} - {np.sum(y_test):.0f} top-10 ({100*np.mean(y_test):.1f}%)")


STEP 4: SPLIT DATA (BEFORE NORMALIZATION)

Splitting: 60% train, 20% val, 20% test

Training set:   (5080, 7) - 3135 top-10 (61.7%)
Validation set: (1694, 7) - 1058 top-10 (62.5%)
Test set:       (1694, 7) - 1077 top-10 (63.6%)


In [8]:
# STEP 5: Compute normalization parameters ONLY from training set
# This prevents data leakage!

print("\n" + "="*70)
print("STEP 5: NORMALIZE (USING TRAINING SET STATISTICS ONLY)")
print("="*70)

print("\nComputing mean and std from TRAINING SET only...")

X_train_mean = X_train_raw.mean(axis=0)
X_train_std = X_train_raw.std(axis=0)

# Avoid division by zero
X_train_std[X_train_std == 0] = 1

print(f"\nNormalization parameters (from {len(X_train_raw)} training samples):")
for i, col in enumerate(feature_columns):
    print(f"  {col:35s}: mean={X_train_mean[i]:8.2f}, std={X_train_std[i]:8.2f}")

# Apply SAME normalization to all sets
print("\nApplying normalization to all sets using TRAINING statistics...")

X_train = (X_train_raw - X_train_mean) / X_train_std
X_val = (X_val_raw - X_train_mean) / X_train_std  # Use TRAIN mean/std!
X_test = (X_test_raw - X_train_mean) / X_train_std  # Use TRAIN mean/std!

print(f"\n✓ Normalization applied to all sets")
print(f"\nVerification - Training set should have mean~0, std~1:")
print(f"  Mean: {X_train.mean(axis=0)}")
print(f"  Std: {X_train.std(axis=0)}")

print(f"\nValidation/Test means should be close to 0 but NOT exactly 0 (expected):")
print(f"  Val mean: {X_val.mean(axis=0)}")
print(f"  Test mean: {X_test.mean(axis=0)}")


STEP 5: NORMALIZE (USING TRAINING SET STATISTICS ONLY)

Computing mean and std from TRAINING SET only...

Normalization parameters (from 5080 training samples):
  laps_completed_early               : mean=   10.00, std=    0.07
  avg_lap_time_relative_zscore       : mean=    0.22, std=    0.71
  lap_time_std                       : mean=27126.32, std=114558.98
  lap_time_min                       : mean=93105.38, std=12960.25
  pace_degradation                   : mean=-18940.09, std=119882.69
  avg_position_early                 : mean=   10.21, std=    5.79
  circuit_normalized                 : mean=    0.03, std=    1.02

Applying normalization to all sets using TRAINING statistics...

✓ Normalization applied to all sets

Verification - Training set should have mean~0, std~1:
  Mean: [-6.6201289e-07  1.9691826e-07 -4.2022450e-08 -1.5315918e-05
  1.7158628e-07 -3.0127098e-06  3.5164394e-08]
  Std: [0.9999092  1.0000002  0.99999994 0.9999999  0.99999905 0.9999982
 1.0000005 ]

Valid

In [9]:
# STEP 6: Save all prepared data for Notebook 04

print("\n" + "="*70)
print("STEP 6: SAVE DATA FOR NOTEBOOK 04")
print("="*70)

# Save normalized numpy arrays
np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/X_val.npy", X_val)
np.save("../data/processed/y_val.npy", y_val)
np.save("../data/processed/X_test.npy", X_test)
np.save("../data/processed/y_test.npy", y_test)

print("\n✓ Data saved as .npy files")

# Save normalization parameters
normalization = {
    'mean': X_train_mean,
    'std': X_train_std,
    'feature_names': feature_columns
}

with open("../data/processed/normalization.pkl", "wb") as f:
    pickle.dump(normalization, f)

print("✓ Normalization parameters saved")

print("\n" + "="*70)
print("DATA PREPARATION COMPLETE!")
print("="*70)

print(f"\nReady for training (Notebook 04):")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

print(f"\nFeatures ({len(feature_columns)}):")
for i, col in enumerate(feature_columns, 1):
    marker = " ← NEW: Z-score relative to circuit!" if col == 'avg_lap_time_relative_zscore' else ""
    print(f"  {i}. {col}{marker}")

print(f"\nKey improvement from previous version:")
print(f"  OLD: avg_lap_time (95s same on Monaco and Monza)")
print(f"  NEW: avg_lap_time_relative_zscore (+0.67 on Monaco vs +6.5 on Monza) ✓")


STEP 6: SAVE DATA FOR NOTEBOOK 04

✓ Data saved as .npy files
✓ Normalization parameters saved

DATA PREPARATION COMPLETE!

Ready for training (Notebook 04):
  X_train: (5080, 7)
  X_val: (1694, 7)
  X_test: (1694, 7)

Features (7):
  1. laps_completed_early
  2. avg_lap_time_relative_zscore ← NEW: Z-score relative to circuit!
  3. lap_time_std
  4. lap_time_min
  5. pace_degradation
  6. avg_position_early
  7. circuit_normalized

Key improvement from previous version:
  OLD: avg_lap_time (95s same on Monaco and Monza)
  NEW: avg_lap_time_relative_zscore (+0.67 on Monaco vs +6.5 on Monza) ✓
